# 1. 16-720 PA2 — Representations and Visual Recognition

This is the executable workflow for Parts B–D. Run it in order, keep all stable labels, and place every graded plot in the PDF. Read `assignment_instructions.pdf` first.

## 2. Run modes

- Default `smoke` mode uses synthetic data for a CPU/API check (no dataset or weight downloads); its synthetic metrics are never performance claims.
- In the first code cell choose `smoke` or `standard` before running setup. Standard mode requires a fresh Colab GPU and uses the exact balanced CIFAR-10 split, 25 epochs, AdamW, cosine decay to `3e-5`, and best-validation checkpoint.
- Do not copy smoke numbers into the report. Record only an actually observed standard run.

In [ ]:
# @title 1. Choose the run mode before setup
# @markdown Keep `smoke` while implementing/testing. Select `standard` only in a fresh GPU runtime.
RUN_MODE = "smoke"  # @param ["smoke", "standard"]
STAFF_REFERENCE = False
if RUN_MODE not in {"smoke", "standard"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'standard'")
print("selected PA2 run mode:", RUN_MODE)

## 3. Academic integrity

Generative AI is prohibited for all of Parts A and B, including written answers, analysis, and `TODO(student)` code. Complete B1 before editing code.

In [ ]:
# SUPPLIED: locate a complete release or upload the course-supplied ZIP in Colab.
from pathlib import Path
import io
import os
import subprocess
import sys
import zipfile

RELEASE_NAME = "16720-26fall-assignment2"
KNOWN_EXTRACT_ROOT = Path(os.environ.get(
    "CV16720_RELEASE_EXTRACT_ROOT", "/content/cv16720_pa2_release"
)).resolve()

def _is_release_root(path):
    required = (
        "requirements-colab.txt", "student/pa2.py", "common/pa2_utils.py",
        "notebooks/pa2_starter.ipynb", "assignment_instructions.pdf",
    )
    return path.is_dir() and all((path / name).is_file() for name in required)

def _single_release_beneath(base):
    if not base.is_dir():
        return None
    roots = sorted({
        path.resolve() for path in (base, *base.iterdir())
        if _is_release_root(path)
    })
    if len(roots) > 1:
        raise RuntimeError(
            f"Found multiple PA2 releases beneath {base}: {roots}. "
            "Use a fresh runtime or select one extracted release."
        )
    return roots[0] if roots else None


def _find_release_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if _is_release_root(candidate):
            return candidate
    return _single_release_beneath(KNOWN_EXTRACT_ROOT)


ROOT = _find_release_root()
if ROOT is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            f"Could not find a complete {RELEASE_NAME} release. Extract the ZIP and "
            "run this notebook from the extracted release root."
        ) from exc

    print("Upload the complete course-supplied pa2_student.zip release.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Upload exactly one pa2_student.zip file, then rerun this cell.")

    extract_root = KNOWN_EXTRACT_ROOT
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            target = (extract_root / member_path).resolve()
            mode = (member.external_attr >> 16) & 0o170000
            if (member_path.is_absolute() or ".." in member_path.parts
                    or (target != extract_root and extract_root not in target.parents)
                    or mode == 0o120000):
                raise RuntimeError(f"Unsafe ZIP member rejected: {member.filename!r}")
        archive.extractall(extract_root)

    ROOT = _single_release_beneath(extract_root)
    if ROOT is None:
        raise RuntimeError(
            f"The uploaded ZIP did not contain a complete flat {RELEASE_NAME} release."
        )

os.chdir(ROOT)
requirements = ROOT / "requirements-colab.txt"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and os.environ.get("CV16720_PA2_COLAB_ROOT") != str(ROOT):
    if not requirements.is_file():
        raise RuntimeError(f"Missing {requirements}; upload the complete student release ZIP.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)
    ])
    os.environ["CV16720_PA2_COLAB_ROOT"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import base64
import csv
import hashlib
import html
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance
from torch.utils.data import DataLoader, Subset, TensorDataset

from common.pa2_utils import (
    CIFAR10_MEAN, CIFAR10_STD, SEED, STANDARD_CONFIG, SMOKE_CONFIG,
    TinyPlainNet, apply_perturbation, classifier_weight_cam,
    cosine_prototypes, count_parameters, hog_embeddings,
    make_offline_smoke_loaders, per_class_recall, plot_confusion_matrix,
    plot_training_curves, prototype_predict, raw_pixel_embeddings,
    seed_everything, build_cifar10_loaders,
)

RUN_MODE = str(globals().get("RUN_MODE", os.environ.get("PA2_RUN_MODE", "smoke"))).strip().lower()
if RUN_MODE not in {"smoke", "standard"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'standard'")
# Install dependencies before checking GPU availability.
if RUN_MODE == "standard" and not torch.cuda.is_available():
    raise RuntimeError("Run the standard configuration on a fresh Colab GPU")
RUN_CONFIG = SMOKE_CONFIG if RUN_MODE == "smoke" else STANDARD_CONFIG
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CIFAR_ROOT = ROOT / "tmp" / "cifar10"
ARTIFACT_DIR = (
    ROOT / "tmp" / "pa2_notebook_smoke"
    if RUN_MODE == "smoke" and bool(globals().get("STAFF_REFERENCE", False))
    else ROOT / "artifacts" / f"pa2_{RUN_MODE}"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED)
print(json.dumps({"root": str(ROOT), "mode": RUN_MODE, "device": str(DEVICE),
                  "torch": torch.__version__, "artifacts": str(ARTIFACT_DIR)}, indent=2))

In [ ]:
def prepare_course_loaders():
    """Use exact CIFAR splits in standard mode and download-free tensors in smoke mode."""
    if RUN_MODE == "standard":
        return build_cifar10_loaders(
            CIFAR_ROOT, mode="standard", download=True, workers=2
        )
    train, test = make_offline_smoke_loaders(train_size=80, test_size=40)
    return train, test, test, [f"class {index}" for index in range(10)]


def collect_loader(loader, limit=None):
    images, labels = [], []
    for batch_images, batch_labels in loader:
        images.append(batch_images.cpu())
        labels.append(batch_labels.to(torch.int64).cpu())
        if limit is not None and sum(part.shape[0] for part in images) >= limit:
            break
    images, labels = torch.cat(images), torch.cat(labels)
    return (images[:limit], labels[:limit]) if limit is not None else (images, labels)


def images_to_rgb01(images):
    """Explicit raw-view path: invert CIFAR normalization before B4/C1/C2."""
    images = torch.as_tensor(images, dtype=torch.float32).cpu()
    if RUN_MODE == "standard":
        mean = torch.tensor(CIFAR10_MEAN).view(1, 3, 1, 1)
        std = torch.tensor(CIFAR10_STD).view(1, 3, 1, 1)
        images = images * std + mean
    return images.clamp(0, 1)


def rgb01_to_model_input(images):
    """Apply the course normalization only after a C2 RGB perturbation."""
    images = torch.as_tensor(images, dtype=torch.float32)
    if RUN_MODE == "standard":
        mean = torch.tensor(CIFAR10_MEAN).view(1, 3, 1, 1)
        std = torch.tensor(CIFAR10_STD).view(1, 3, 1, 1)
        images = (images - mean) / std
    return images


def make_fixed_train_eval_loader(train_loader, val_loader):
    """Return released training indices with fixed evaluation preprocessing/order.

    The standard validation loader's base CIFAR dataset uses the non-augmented
    evaluation transform. Reusing it with the training subset's released indices
    prevents stochastic crops/flips and keeps all C1 feature rows aligned.
    """
    if RUN_MODE == "standard":
        train_subset = train_loader.dataset
        validation_subset = val_loader.dataset
        if not isinstance(train_subset, Subset) or not isinstance(validation_subset, Subset):
            raise TypeError("standard loaders must use Subset datasets")
        fixed_dataset = Subset(validation_subset.dataset, list(train_subset.indices))
    else:
        images, labels = collect_loader(train_loader)
        fixed_dataset = TensorDataset(images, labels)
    return DataLoader(
        fixed_dataset, batch_size=RUN_CONFIG.batch_size, shuffle=False, num_workers=0
    )


def clone_loader_stream(source_loader, *, shuffle, seed=SEED):
    """Clone a released loader with independent, reset sampler/worker RNG state."""
    workers = source_loader.num_workers
    return DataLoader(
        source_loader.dataset,
        batch_size=source_loader.batch_size,
        shuffle=shuffle,
        num_workers=workers,
        collate_fn=source_loader.collate_fn,
        pin_memory=source_loader.pin_memory,
        drop_last=source_loader.drop_last,
        worker_init_fn=source_loader.worker_init_fn,
        generator=torch.Generator().manual_seed(seed),
        persistent_workers=workers > 0,
    )


def make_c3_controlled_loaders(source_train_loader, source_val_loader):
    """Create a fresh C3 data stream; call once independently per architecture."""
    return (
        clone_loader_stream(
            source_train_loader, shuffle=(RUN_MODE == "standard"), seed=SEED
        ),
        clone_loader_stream(source_val_loader, shuffle=False, seed=SEED),
    )


def verify_c3_stream_reset(source_train_loader, source_val_loader):
    """Probe separate clones so the two actual comparison streams stay untouched."""
    left, _ = make_c3_controlled_loaders(source_train_loader, source_val_loader)
    right, _ = make_c3_controlled_loaders(source_train_loader, source_val_loader)
    generator_equal = torch.equal(left.generator.get_state(), right.generator.get_state())
    left_images, left_labels = next(iter(left))
    right_images, right_labels = next(iter(right))
    images_equal = torch.equal(left_images, right_images)
    labels_equal = torch.equal(left_labels, right_labels)
    if not (generator_equal and images_equal and labels_equal):
        raise AssertionError("C3 loader sampler/worker augmentation streams did not reset")
    return {
        "seed": SEED,
        "generator_state_equal": generator_equal,
        "first_augmented_batch_equal": images_equal,
        "first_label_batch_equal": labels_equal,
        "batch_size": int(left_labels.numel()),
        "workers_per_loader": left.num_workers,
    }


def distinct_query_indices(labels, count=3):
    selected, seen = [], set()
    for index, label in enumerate(torch.as_tensor(labels).tolist()):
        if label not in seen:
            selected.append(index); seen.add(label)
        if len(selected) == count:
            return selected
    raise ValueError("not enough distinct classes")


def plot_architecture_evidence(model, image):
    captured = {}
    hooks = [
        model.stage1.register_forward_hook(
            lambda _module, _inputs, output: captured.update(stage1=output.detach().cpu())
        ),
        model.stage2.register_forward_hook(
            lambda _module, _inputs, output: captured.update(stage2=output.detach().cpu())
        ),
    ]
    model.eval()
    with torch.no_grad():
        model(image[:1].to(DEVICE))
    for hook in hooks:
        hook.remove()
    weights = model.stem.conv.weight.detach().cpu()[:8]
    figure, axes = plt.subplots(3, 8, figsize=(14, 5.5))
    for column, kernel in enumerate(weights):
        view = kernel.permute(1, 2, 0)
        view = (view - view.min()) / (view.max() - view.min() + 1e-12)
        axes[0, column].imshow(view); axes[0, column].set_title(f"filter {column}", fontsize=8)
        axes[1, column].imshow(captured["stage1"][0, column], cmap="magma")
        axes[2, column].imshow(captured["stage2"][0, column], cmap="magma")
    axes[1, 0].set_ylabel("stage 1"); axes[2, 0].set_ylabel("stage 2")
    for axis in axes.flat:
        axis.set_xticks([]); axis.set_yticks([])
    figure.suptitle("B2 untrained filters and feature maps (independent display scales)")
    figure.tight_layout()
    return figure


def plot_retrieval(images, labels, feature_sets, query_indices, nearest_fn, k=5):
    """One fixed gallery and the same three queries for every representation."""
    rgb = images_to_rgb01(images)
    rows, columns = len(feature_sets), len(query_indices) * (k + 1)
    figure, axes = plt.subplots(rows, columns, figsize=(2 * columns, 2.1 * rows), squeeze=False)
    for row, (name, features) in enumerate(feature_sets.items()):
        for group, query_index in enumerate(query_indices):
            start = group * (k + 1)
            axes[row, start].imshow(rgb[query_index].permute(1, 2, 0))
            axes[row, start].set_title(f"query\ny={int(labels[query_index])}", fontsize=8)
            for spine in axes[row, start].spines.values():
                spine.set_color("#d62728"); spine.set_linewidth(2)
            indices, scores = nearest_fn(features[query_index], features, k,
                                         exclude_index=query_index)
            for offset, (index, score) in enumerate(zip(indices.tolist(), scores.tolist()), 1):
                axes[row, start + offset].imshow(rgb[index].permute(1, 2, 0))
                axes[row, start + offset].set_title(
                    f"y={int(labels[index])}\ncos={score:.2f}", fontsize=8
                )
        axes[row, 0].set_ylabel(name)
    for axis in axes.flat:
        axis.set_xticks([]); axis.set_yticks([])
    figure.suptitle("B4 common queries: query plus five self-excluded neighbors")
    figure.tight_layout()
    return figure

def save_course_figure(figure, filename):
    target = ARTIFACT_DIR / filename
    figure.savefig(target, dpi=150, bbox_inches="tight")
    print("Saved figure:", target)
    plt.show()
    return figure


In [ ]:
def train_with_checkpoint(model, train_loader, val_loader, train_fn, eval_fn, *, epochs, path):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=3e-5
    )
    history = {name: [] for name in (
        "train_loss", "train_accuracy", "val_loss", "val_accuracy", "learning_rate"
    )}
    best_accuracy = -math.inf
    for epoch in range(epochs):
        train_loss, train_accuracy = train_fn(model, train_loader, optimizer, DEVICE)
        val_loss, val_accuracy, *_ = eval_fn(model, val_loader, DEVICE)
        for key, value in (
            ("train_loss", train_loss), ("train_accuracy", train_accuracy),
            ("val_loss", val_loss), ("val_accuracy", val_accuracy),
            ("learning_rate", optimizer.param_groups[0]["lr"]),
        ):
            history[key].append(value)
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            torch.save({
                "state_dict": model.state_dict(),
                "metadata": {"mode": RUN_MODE, "seed": SEED, "epoch": epoch + 1,
                             "validation_accuracy": float(val_accuracy),
                             "architecture": "TinyResNet-[2,2,2]"},
            }, path)
        scheduler.step()
        print(f"epoch {epoch + 1:02d}/{epochs}: train={train_loss:.4f}/{train_accuracy:.3f}, "
              f"val={val_loss:.4f}/{val_accuracy:.3f}")
    payload = torch.load(path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(payload["state_dict"])
    return history, payload["metadata"]


def checkpoint_reload_is_identical(model_factory, checkpoint_path, model, sample):
    reloaded = model_factory().to(DEVICE)
    payload = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    reloaded.load_state_dict(payload["state_dict"])
    model.eval(); reloaded.eval()
    with torch.no_grad():
        expected = model(sample.to(DEVICE)).cpu()
        actual = reloaded(sample.to(DEVICE)).cpu()
    return torch.equal(expected, actual), reloaded


def cosine_knn_predict(train_features, train_labels, query_features, k=5, *, batch_size=256):
    """Exact C1 cosine voting with stable ties and bounded query-batch memory.
    """
    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size <= 0:
        raise ValueError("batch_size must be a positive integer")
    train = torch.as_tensor(train_features, dtype=torch.float32).detach().cpu()
    query = torch.as_tensor(query_features, dtype=torch.float32).detach().cpu()
    labels = torch.as_tensor(train_labels, dtype=torch.int64).detach().cpu()
    if train.ndim != 2 or query.ndim != 2 or train.shape[1] != query.shape[1]:
        raise ValueError("train/query features must be matrices with matching feature dimensions")
    if labels.ndim != 1 or labels.numel() != train.shape[0] or labels.numel() == 0:
        raise ValueError("train_labels must contain one label per training image")
    if bool((labels < 0).any()):
        raise ValueError("class labels must be nonnegative")
    if not isinstance(k, int) or isinstance(k, bool) or not 1 <= k <= len(train):
        raise ValueError("k must be an integer between 1 and the training-gallery size")
    if not bool(torch.isfinite(train).all()) or not bool(torch.isfinite(query).all()):
        raise ValueError("features must be finite")
    train = F.normalize(train, dim=1)
    query = F.normalize(query, dim=1)
    neighbors = torch.empty((len(query), k), dtype=torch.int64)
    predictions = torch.empty(len(query), dtype=torch.int64)
    classes = int(labels.max()) + 1
    for start in range(0, len(query), batch_size):
        stop = min(start + batch_size, len(query))
        similarity = query[start:stop] @ train.T
        selected = torch.argsort(
            similarity, dim=1, descending=True, stable=True
        )[:, :k].clone()
        del similarity
        neighbors[start:stop] = selected
        counts = torch.zeros((stop - start, classes), dtype=torch.int64)
        counts.scatter_add_(1, labels[selected], torch.ones_like(selected))
        predictions[start:stop] = counts.argmax(dim=1)
    return predictions, neighbors


def representation_metrics(train_features, train_labels, test_features, test_labels, classes=10):
    prediction, neighbors = cosine_knn_predict(train_features, train_labels,
                                                test_features, k=5)
    confusion = torch.zeros((classes, classes), dtype=torch.int64)
    for truth, guess in zip(test_labels.tolist(), prediction.tolist()):
        confusion[truth, guess] += 1
    return {"accuracy": float((prediction == test_labels).float().mean()),
            "recall": per_class_recall(confusion), "prediction": prediction,
            "neighbors": neighbors}


def plot_representation_metrics(reports, class_names):
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    names = list(reports)
    axes[0].bar(names, [reports[name]["accuracy"] for name in names], color="#4c78a8")
    axes[0].set(title="Deterministic cosine 5-NN", ylabel="accuracy", ylim=(0, 1))
    recalls = torch.stack([reports[name]["recall"] for name in names]).numpy()
    image = axes[1].imshow(recalls, vmin=0, vmax=1, cmap="Blues", aspect="auto")
    axes[1].set(title="Per-class recall", yticks=range(len(names)), yticklabels=names,
                xticks=range(len(class_names),), xticklabels=class_names)
    plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right")
    figure.colorbar(image, ax=axes[1]); figure.tight_layout()
    return figure


def nuisance_sweep(model, images, labels, plans):
    """Apply nuisances in RGB [0,1], then normalize once before inference."""
    rgb, records = images_to_rgb01(images), {}
    model.eval()
    for kind, severities in plans.items():
        rows = []
        for severity in severities:
            changed = np.stack([
                apply_perturbation(image.permute(1, 2, 0).numpy(), kind,
                                   float(severity), seed=SEED + index)
                for index, image in enumerate(rgb)
            ])
            changed_tensor = torch.from_numpy(changed).permute(0, 3, 1, 2)
            with torch.no_grad():
                probability = torch.softmax(model(
                    rgb01_to_model_input(changed_tensor).to(DEVICE)
                ), dim=1)
            confidence, prediction = probability.max(1)
            rows.append({"severity": float(severity),
                         "accuracy": float((prediction.cpu() == labels).float().mean()),
                         "confidence": float(confidence.mean().cpu()),
                         "example": changed[0]})
        records[kind] = rows
    return records


def plot_nuisance_records(records):
    figure, axes = plt.subplots(2, len(records), figsize=(4 * len(records), 6), squeeze=False)
    for column, (kind, rows) in enumerate(records.items()):
        severity = [row["severity"] for row in rows]
        axes[0, column].plot(severity, [row["accuracy"] for row in rows], "o-", label="accuracy")
        axes[0, column].plot(severity, [row["confidence"] for row in rows], "s--",
                             label="mean confidence")
        axes[0, column].set(title=kind, xlabel="severity", ylim=(0, 1))
        axes[0, column].grid(alpha=.25); axes[0, column].legend(fontsize=8)
        axes[1, column].imshow(rows[-1]["example"])
        axes[1, column].set_title(f"max severity={rows[-1]['severity']:g}")
        axes[1, column].axis("off")
    figure.suptitle("C2 pre-normalization nuisance sweeps")
    figure.tight_layout()
    return figure

In [ ]:
ATLAS_FIELDS = ("filename", "category", "split", "own_photo",
                "source_url_or_creator", "license_or_permission", "consent_privacy_note")
ATLAS_IMAGE_MIME = {
    ".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
    ".webp": "image/webp",
}


def validate_completed_approval_record(path):
    """Require staff identity/date/basis/package record and an explicit waiver."""
    path = Path(path)
    if not path.is_file():
        return {"ok": False, "errors": [f"missing staff approval record: {path}"]}
    text = path.read_text(encoding="utf-8")
    required = (
        "Student identifier", "Approving staff member", "Approval date", "Basis",
        "Original-photo rule waived", "Approved package SHA-256 manifest",
    )
    values, errors = {}, []
    for field in required:
        prefix = f"- {field}:"
        line = next((line for line in text.splitlines() if line.startswith(prefix)), "")
        value = line[len(prefix):].strip() if line else ""
        values[field] = value
        if not value:
            errors.append(f"approval record has no {field}")
    if values.get("Original-photo rule waived", "").lower() != "yes":
        errors.append("fallback use requires 'Original-photo rule waived: yes'")
    return {"ok": not errors, "errors": errors, "values": values, "path": str(path)}


def read_and_validate_manifest(path, image_dir, *, enforce_own_photo=True):
    path, image_dir = Path(path), Path(image_dir)
    if not path.exists():
        return {"ok": False, "errors": [f"missing manifest: {path}"], "rows": []}
    errors = []
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        missing = [field for field in ATLAS_FIELDS if field not in (reader.fieldnames or [])]
        if missing: errors.append(f"missing fields: {missing}")
        rows = list(reader)
    categories = sorted({row.get("category", "").strip() for row in rows
                         if row.get("category", "").strip()})
    if len(categories) != 3:
        errors.append(f"expected exactly three categories, found {len(categories)}")
    image_root = image_dir.resolve()
    seen_names, seen_digests = {}, {}
    for row in rows:
        filename = row.get("filename", "")
        candidate_name = Path(filename)
        safe_basename = bool(filename) and filename == filename.strip()
        safe_basename = safe_basename and not candidate_name.is_absolute()
        safe_basename = safe_basename and "/" not in filename and "\\" not in filename
        safe_basename = safe_basename and candidate_name.name not in {".", ".."}
        if not safe_basename:
            errors.append(f"filename must be one relative basename: {filename!r}")
        suffix = candidate_name.suffix.lower()
        if suffix not in ATLAS_IMAGE_MIME:
            errors.append(f"unsupported image suffix for {filename!r}")
        name_key = filename.casefold()
        if name_key in seen_names:
            errors.append(f"duplicate filename: {filename!r} also used by {seen_names[name_key]!r}")
        else:
            seen_names[name_key] = filename
        if safe_basename and suffix in ATLAS_IMAGE_MIME:
            candidate = image_dir / filename
            resolved = candidate.resolve()
            inside_root = resolved.parent == image_root
            if candidate.is_symlink() or not inside_root:
                errors.append(f"image must be a regular file inside image_dir: {filename!r}")
            elif not candidate.is_file():
                errors.append(f"missing image: {filename!r}")
            else:
                digest = hashlib.sha256(candidate.read_bytes()).hexdigest()
                if digest in seen_digests:
                    errors.append(
                        f"duplicate image content: {filename!r} matches {seen_digests[digest]!r}"
                    )
                else:
                    seen_digests[digest] = filename
        if row.get("split") not in {"development", "held-out"}:
            errors.append(f"invalid split: {filename!r}")
        for field in ATLAS_FIELDS[4:]:
            if not row.get(field, "").strip():
                errors.append(f"{filename!r} has empty {field}")
    for category in categories:
        selected = [row for row in rows if row.get("category", "").strip() == category]
        if not 8 <= len(selected) <= 15:
            errors.append(f"{category!r} needs 8-15 images, found {len(selected)}")
        if not {"development", "held-out"}.issubset({row.get("split") for row in selected}):
            errors.append(f"{category!r} needs both splits")
        own = sum(row.get("own_photo", "").strip().lower() == "true" for row in selected)
        if enforce_own_photo and own < math.ceil(len(selected) / 2):
            errors.append(f"{category!r} needs at least half own photographs, found {own}/{len(selected)}")
    return {"ok": not errors, "errors": errors, "rows": rows, "categories": categories}


def make_atlas_extractor():
    """Return model plus its exact weight-enum preprocessing contract."""
    from torchvision import models, transforms
    if RUN_MODE == "standard":
        weights = models.ResNet18_Weights.DEFAULT
        extractor = models.resnet18(weights=weights)
        preprocess = weights.transforms()  # preprocessing supplied with these weights
        weight_note = f"torchvision {weights.__class__.__name__}.{weights.name}"
    else:
        extractor = models.resnet18(weights=None)
        preprocess = transforms.Compose([
            transforms.Resize((96, 96)), transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ])
        weight_note = "random ResNet18 weights: smoke test only"
    dimension = extractor.fc.in_features
    extractor.fc = torch.nn.Identity()
    extractor.eval()
    for parameter in extractor.parameters():
        parameter.requires_grad_(False)
    return extractor.to(DEVICE), dimension, weight_note, preprocess


def load_atlas_images(rows, image_dir, preprocess, *, rgb_intervention=None):
    tensors, labels, paths, splits = [], [], [], []
    categories = sorted({row["category"] for row in rows})
    category_to_index = {name: index for index, name in enumerate(categories)}
    for row in rows:
        path = Path(image_dir) / row["filename"]
        with Image.open(path) as image:
            rgb = image.convert("RGB")
            if rgb_intervention is not None:
                rgb = rgb_intervention(rgb)
            tensors.append(preprocess(rgb))
        labels.append(category_to_index[row["category"]]); paths.append(path)
        splits.append(row["split"])
    return torch.stack(tensors), torch.tensor(labels), paths, splits, categories


def extract_frozen_features(extractor, tensors, batch_size=16):
    result = []
    extractor.eval()
    with torch.no_grad():
        for start in range(0, len(tensors), batch_size):
            result.append(extractor(tensors[start:start + batch_size].to(DEVICE)).cpu())
    return torch.cat(result)


def image_data_uri(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix not in ATLAS_IMAGE_MIME:
        raise ValueError(f"unsupported embedded image suffix: {suffix!r}")
    return (f"data:{ATLAS_IMAGE_MIME[suffix]};base64,"
            + base64.b64encode(path.read_bytes()).decode("ascii"))


def select_atlas_cases(held_rows, predictions, scores, categories):
    if not held_rows or len(predictions) != len(held_rows) or len(scores) != len(held_rows):
        raise ValueError("Atlas case selection requires aligned, nonempty held-out results")
    correct = torch.tensor([
        row["category"] == categories[int(prediction)]
        for row, prediction in zip(held_rows, predictions)
    ])
    if bool(correct.any()):
        candidates = torch.nonzero(correct, as_tuple=False).flatten()
        success = int(candidates[torch.as_tensor(scores)[candidates].argmax()])
    else:
        success = None 
    if bool((~correct).any()):
        candidates = torch.nonzero(~correct, as_tuple=False).flatten()
        evidence = int(candidates[torch.as_tensor(scores)[candidates].argmax()])
        evidence_kind = "highest-score held-out error"
    else:
        evidence = int(torch.as_tensor(scores).argmin())
        evidence_kind = "lowest-score boundary case; no held-out error"
    return success, evidence, evidence_kind


def plot_atlas_gallery(paths, rows, predictions, scores, categories, output_path):
    count = min(9, len(paths))
    figure, axes = plt.subplots(3, 3, figsize=(9, 9), squeeze=False)
    for axis, path, row, prediction, score in zip(
        axes.flat, paths[:count], rows[:count], predictions[:count], scores[:count]
    ):
        with Image.open(path) as image: axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"true={row['category']}\npred={categories[int(prediction)]}; cos={float(score):.3f}",
            fontsize=8,
        )
        axis.axis("off")
    for axis in axes.flat[count:]: axis.axis("off")
    figure.suptitle("Held-out gallery: truth, prediction, uncalibrated cosine")
    figure.tight_layout(); figure.savefig(output_path, dpi=150, bbox_inches="tight")
    return figure


def plot_atlas_retrieval(query_path, query_row, query_feature, development_paths,
                         development_rows, development_features, output_path, k=5):
    query = F.normalize(torch.as_tensor(query_feature, dtype=torch.float32), dim=0)
    gallery = F.normalize(torch.as_tensor(development_features, dtype=torch.float32), dim=1)
    similarity = gallery @ query
    order = torch.argsort(similarity, descending=True, stable=True)[:k]
    figure, axes = plt.subplots(1, k + 1, figsize=(2.25 * (k + 1), 2.5))
    with Image.open(query_path) as image: axes[0].imshow(image.convert("RGB"))
    axes[0].set_title(f"held-out query\ntrue={query_row['category']}", fontsize=8)
    for spine in axes[0].spines.values(): spine.set_color("#d62728"); spine.set_linewidth(2)
    for column, index in enumerate(order.tolist(), 1):
        with Image.open(development_paths[index]) as image:
            axes[column].imshow(image.convert("RGB"))
        axes[column].set_title(
            f"dev: {development_rows[index]['category']}\ncos={float(similarity[index]):.3f}",
            fontsize=8,
        )
    for axis in axes: axis.set_xticks([]); axis.set_yticks([])
    figure.suptitle("Failure/boundary evidence: development-set retrieval")
    figure.tight_layout(); figure.savefig(output_path, dpi=150, bbox_inches="tight")
    return figure, order, similarity[order]


def plot_atlas_prototype_cam(extractor, input_tensor, raw_path, prototype,
                             predicted_category, output_path, *, preprocess):
    """Overlay prototype CAM on the exact input crop, not the full photograph."""
    if not hasattr(extractor, "layer4"):
        raise TypeError("released Atlas CAM helper requires the approved ResNet18 path")
    # Weight presets expose mean/std directly; smoke Compose contains Normalize.
    normalization = preprocess
    if not (hasattr(normalization, "mean") and hasattr(normalization, "std")):
        normalization = next((
            transform for transform in reversed(getattr(preprocess, "transforms", []))
            if hasattr(transform, "mean") and hasattr(transform, "std")
        ), None)
    if normalization is None or not hasattr(normalization, "mean") or not hasattr(normalization, "std"):
        raise ValueError("Pass the exact model preprocessing with its normalization mean/std")
    display_tensor = input_tensor.detach().float().cpu()
    mean = torch.as_tensor(normalization.mean, dtype=torch.float32).view(-1, 1, 1)
    std = torch.as_tensor(normalization.std, dtype=torch.float32).view(-1, 1, 1)
    rgb = (display_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    captured = {}
    hook = extractor.layer4.register_forward_hook(
        lambda _module, _inputs, output: captured.update(feature_map=output.detach())
    )
    extractor.eval()
    try:
        with torch.no_grad():
            extractor(input_tensor[None].to(DEVICE))
    finally:
        hook.remove()
    feature_map = captured["feature_map"][0]
    weight = F.normalize(torch.as_tensor(prototype, device=feature_map.device), dim=0)
    cam = torch.einsum("c,chw->hw", weight, feature_map).relu()
    cam = cam - cam.min(); cam = cam / cam.max().clamp_min(1e-12)
    cam = F.interpolate(cam[None, None], size=rgb.shape[:2], mode="bilinear",
                        align_corners=False)[0, 0].cpu()
    figure, axes = plt.subplots(1, 2, figsize=(7, 3.3))
    axes[0].imshow(rgb); axes[0].set_title("model input after resize/crop")
    axes[1].imshow(rgb); axes[1].imshow(cam, cmap="jet", alpha=.45)
    axes[1].set_title(f"prototype CAM: {predicted_category}\ncorrelational, not causal")
    for axis in axes: axis.axis("off")
    figure.tight_layout(); figure.savefig(output_path, dpi=150, bbox_inches="tight")
    return figure


def plot_atlas_intervention(paths, rows, categories, before_predictions, before_scores,
                            after_predictions, after_scores, intervention, output_path):
    selected = []
    for category in categories:
        matches = [index for index, row in enumerate(rows) if row["category"] == category]
        if matches: selected.append(matches[0])
    figure, axes = plt.subplots(2, len(selected), figsize=(3.2 * len(selected), 6), squeeze=False)
    for column, index in enumerate(selected):
        with Image.open(paths[index]) as image: before = image.convert("RGB")
        after = intervention(before.copy())
        axes[0, column].imshow(before)
        axes[0, column].set_title(
            f"before: {categories[int(before_predictions[index])]}\ncos={float(before_scores[index]):.3f}",
            fontsize=8,
        )
        axes[1, column].imshow(after)
        axes[1, column].set_title(
            f"after: {categories[int(after_predictions[index])]}\ncos={float(after_scores[index]):.3f}",
            fontsize=8,
        )
    for axis in axes.flat: axis.axis("off")
    figure.suptitle("Measured held-out before/after intervention")
    figure.tight_layout(); figure.savefig(output_path, dpi=150, bbox_inches="tight")
    return figure


def embedded_figure(path, alt, caption):
    return ("<figure><img alt='" + html.escape(alt, quote=True) + "' src='"
            + image_data_uri(path) + "'><figcaption>"
            + html.escape(caption) + "</figcaption></figure>")


def export_atlas_html(template_path, output_path, *, categories, held_rows, held_paths,
                      predictions, scores, weight_note, gallery_path, retrieval_path,
                      cam_path, intervention_path, retrieval_caption, cam_caption,
                      intervention_summary, limitation):
    for asset in (gallery_path, retrieval_path, cam_path, intervention_path):
        if not Path(asset).is_file() or Path(asset).stat().st_size == 0:
            raise FileNotFoundError(f"missing generated Atlas evidence asset: {asset}")
    cards = []
    for row, path, prediction, score in zip(held_rows, held_paths, predictions, scores):
        cards.append("<figure><img alt='held-out example' src='" + image_data_uri(path) + "'>"
                     f"<figcaption>true={html.escape(row['category'])}; "
                     f"pred={html.escape(categories[int(prediction)])}; "
                     f"uncalibrated cosine={float(score):.3f}</figcaption></figure>")
    success_index, _, _ = select_atlas_cases(held_rows, predictions, scores, categories)
    if success_index is None:
        success_card = (
            '<p data-evidence="no-success">No held-out success was observed.</p>'
        )
    else:
        success_row = held_rows[success_index]
        success_prediction = categories[int(predictions[success_index])]
        success_card = (
            '<figure data-evidence="selected-success"><img alt="Selected clear success" src="'
            + image_data_uri(held_paths[success_index]) + '"><figcaption>'
            + html.escape(
                f"Clear success: true={success_row['category']}; pred={success_prediction}; "
                f"uncalibrated cosine={float(scores[success_index]):.3f}"
            )
            + "</figcaption></figure>"
        )
    page = Path(template_path).read_text(encoding="utf-8")
    replacements = {
        "[three-category world]": html.escape(" / ".join(categories)),
        "[One-sentence question and model description.]": html.escape(
            "Can a frozen representation distinguish these categories? " + weight_note),
        "[Replace with labeled figures.]": success_card + embedded_figure(
            gallery_path, "Held-out gallery", "Held-out predictions and uncalibrated cosine scores."
        ) + "".join(cards),
        "[Feature-space or retrieval view.]": embedded_figure(
            retrieval_path, "Development-set nearest-neighbor retrieval", retrieval_caption
        ),
        "[Success, failure, CAM/occlusion/retrieval evidence.]": embedded_figure(
            cam_path, "Prototype classifier-weight CAM evidence", cam_caption
        ),
        "[Before/after change and held-out effect.]": embedded_figure(
            intervention_path, "Measured before and after intervention", intervention_summary
        ),
        "[What the result assumes and does not establish.]": html.escape(limitation),
    }
    for marker, value in replacements.items(): page = page.replace(marker, value)
    unresolved = [marker for marker in replacements if marker in page]
    if unresolved or "[replace" in page.lower():
        raise ValueError(f"unresolved Atlas replacement markers: {unresolved}")
    Path(output_path).write_text(page, encoding="utf-8")
    return Path(output_path)


def plot_atlas_hero(paths, rows, predictions, scores, categories, output_path):
    # Keep all three categories visible instead of taking a manifest-order prefix.
    selected = []
    for category in categories:
        matches = [index for index, row in enumerate(rows)
                   if row["category"] == category]
        selected.extend(matches[:2])
    selected = selected[:6]
    count = len(selected)
    figure, axes = plt.subplots(2, 3, figsize=(10, 6))
    for axis, index in zip(axes.flat, selected):
        path, row = paths[index], rows[index]
        prediction, score = predictions[index], scores[index]
        with Image.open(path) as image: axis.imshow(image.convert("RGB"))
        axis.set_title(f"true: {row['category']}\npred: {categories[int(prediction)]} ({float(score):.2f})",
                       fontsize=9)
        axis.axis("off")
    for axis in axes.flat[count:]: axis.axis("off")
    qualifier = "smoke test only" if RUN_MODE == "smoke" else "held-out prototype results"
    figure.suptitle(f"Recognition Atlas — {qualifier}")
    figure.tight_layout(); figure.savefig(output_path, dpi=160, bbox_inches="tight")
    return figure

## 4. Released conventions

Tensors are `(B,C,H,W)`; logits enter cross-entropy; confusion rows are truth; metrics are sample-weighted; retrieval uses cosine similarity and stable ties; evaluation uses `eval()` plus no gradients; nuisance changes occur in RGB before normalization.

In [ ]:
train_loader, val_loader, test_loader, class_names = prepare_course_loaders()
print({"mode": RUN_MODE, "train": len(train_loader.dataset),
       "validation": len(val_loader.dataset), "test": len(test_loader.dataset),
       "classes": class_names})

## 5. B1 — Read the pipeline before implementation

1. **[B1-Q1]** Trace batch, stages, final map, embedding, and logits.
2. **[B1-Q2]** Identify training-only transforms and shared normalization.
3. **[B1-Q3]** State what `train()`/`eval()` change for BatchNorm.
4. **[B1-Q4]** Locate logits entering loss and softmax used for confidence.
5. **[B1-Q5]** Trace `forward_features`; explain why it excludes the classifier.

### 5.1. Student evidence

Put all five labeled answers in the PDF. Most likely pipeline assumption to violate: **[replace]**.

## 6. B2 — Residual classifier and visual evidence

In Colab's Files pane, open `student/pa2.py`, edit only its marked regions, and
save (Ctrl+S / Cmd+S). Run the reload cell below after each edit, then rerun
dependent cells to recreate models and results. Run the public-test cell after B4
once the TODOs are complete; its coverage is partial.


In [ ]:
# SUPPLIED: reload saved implementation changes before recreating models.
import importlib
import student.pa2 as pa2
pa2 = importlib.reload(pa2)
from student.pa2 import (
    ConvNormAct, ResidualBlock, TinyResNet, train_one_epoch, evaluate,
    compute_confusion_matrix, extract_embeddings, nearest_neighbors,
)
print("Loaded:", pa2.__file__)


In [ ]:
# TODO(student): implement only the marked regions in student/pa2.py.
model = TinyResNet(10).to(DEVICE)
# Record stage shapes from the same forward pass used for the final trace.
stage_shapes = {}
def record_stage(name):
    def hook(_module, _inputs, output):
        stage_shapes[name] = tuple(output.shape)
    return hook
trace_hooks = [getattr(model, name).register_forward_hook(record_stage(name))
               for name in ("stem", "stage1", "stage2", "stage3")]
model.eval()
trace_input = torch.randn(3, 3, 32, 32, device=DEVICE)
with torch.no_grad():
    feature_map, embedding = model.forward_features(trace_input)
for hook in trace_hooks:
    hook.remove()
print("Stage shapes:", stage_shapes)
trace = {"feature_map": tuple(feature_map.shape), "embedding": tuple(embedding.shape),
         "logits": tuple(model(trace_input).shape), "parameters": count_parameters(model)}
print(trace)
assert trace == {"feature_map": (3, 128, 8, 8), "embedding": (3, 128),
                 "logits": (3, 10), "parameters": 696_618}

In [ ]:
b2_images, _ = collect_loader(test_loader, 1)
save_course_figure(plot_architecture_evidence(model, b2_images), "b2_architecture.png")

### 6.1. Student evidence — B2

Include the trace/count, eight filters, and two-stage maps. Explain one visible pattern without claiming that a lone channel is a semantic detector: **[replace]**.

## 7. B3 — Train, evaluate, checkpoint, curves, and confusion

In [ ]:
checkpoint_path = ARTIFACT_DIR / f"tinyresnet_{RUN_MODE}.pt"
epochs = 1 if RUN_MODE == "smoke" else 25
history, checkpoint_metadata = train_with_checkpoint(
    model, train_loader, val_loader, train_one_epoch, evaluate,
    epochs=epochs, path=checkpoint_path)
test_loss, test_accuracy, test_predictions, test_labels, test_confidence = evaluate(
    model, test_loader, DEVICE)
test_confusion = compute_confusion_matrix(test_labels, test_predictions, 10)
print({"checkpoint": str(checkpoint_path), "metadata": checkpoint_metadata,
       "test_loss": test_loss, "test_accuracy": test_accuracy,
       "confusion_total": int(test_confusion.sum())})

In [ ]:
reload_sample, _ = collect_loader(test_loader, 4)
identical, reloaded_model = checkpoint_reload_is_identical(
    lambda: TinyResNet(10), checkpoint_path, model, reload_sample)
assert identical
print("checkpoint reload logits exactly identical:", identical)
curve_figure, _ = plot_training_curves(history)
save_course_figure(curve_figure, "b3_training_curves.png")
confusion_figure, _ = plot_confusion_matrix(test_confusion, class_names)
save_course_figure(confusion_figure, "b3_confusion.png")

### 7.1. Student evidence — B3

Report observed standard duration, best epoch, final test result, curve interpretation, and two confusion patterns. Confirm checkpoint contents: **[replace]**.

## 8. B4 — Raw, untrained, and trained retrieval

In [ ]:
b4_images, b4_labels = collect_loader(test_loader)
# This is an explicit inverse-normalized raw [0,1] gallery, not standardized pixels.
b4_raw = raw_pixel_embeddings(images_to_rgb01(b4_images))
seed_everything(SEED + 1)
untrained_model = TinyResNet(10).to(DEVICE)
b4_untrained, _ = extract_embeddings(untrained_model, test_loader, DEVICE)
b4_trained, _ = extract_embeddings(model, test_loader, DEVICE)
seed_everything(SEED)
b4_query_indices = distinct_query_indices(b4_labels, 3)
b4_features = {"normalized raw pixels": b4_raw, "untrained TinyResNet": b4_untrained,
               "trained TinyResNet": b4_trained}
b4_figure = plot_retrieval(b4_images, b4_labels, b4_features, b4_query_indices,
                           nearest_neighbors, 5)
save_course_figure(b4_figure, "b4_retrieval.png")

### 8.1. Student evidence — B4

Annotate each common-query neighborhood's apparent organizing property and state one claim the panel cannot establish: **[replace]**.

In [ ]:
# SUPPLIED: expected to fail until the student TODOs are implemented.
test_environment = os.environ.copy()
test_environment["PA2_MODULE"] = "student.pa2"
subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_pa2_public.py", "-q"],
    cwd=ROOT, env=test_environment, check=True,
)


## 9. C1 — Controlled HOG/random/trained comparison

In [ ]:
c1_train_fixed = make_fixed_train_eval_loader(train_loader, val_loader)
c1_train_images, c1_train_labels = collect_loader(c1_train_fixed)
c1_test_images, c1_test_labels = collect_loader(test_loader)
c1_test_fixed = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(c1_test_images, c1_test_labels),
    batch_size=RUN_CONFIG.batch_size, shuffle=False)
hog_train = torch.from_numpy(hog_embeddings(
    images_to_rgb01(c1_train_images).permute(0, 2, 3, 1).numpy()))
hog_test = torch.from_numpy(hog_embeddings(
    images_to_rgb01(c1_test_images).permute(0, 2, 3, 1).numpy()))
random_train, random_train_labels = extract_embeddings(untrained_model, c1_train_fixed, DEVICE)
random_test, random_test_labels = extract_embeddings(untrained_model, c1_test_fixed, DEVICE)
trained_train, trained_train_labels = extract_embeddings(model, c1_train_fixed, DEVICE)
trained_test, trained_test_labels = extract_embeddings(model, c1_test_fixed, DEVICE)
assert torch.equal(random_train_labels, c1_train_labels)
assert torch.equal(random_test_labels, c1_test_labels)
assert torch.equal(trained_train_labels, c1_train_labels)
assert torch.equal(trained_test_labels, c1_test_labels)
c1_reports = {
    "HOG": representation_metrics(hog_train, c1_train_labels, hog_test, c1_test_labels),
    "random CNN": representation_metrics(random_train, c1_train_labels, random_test, c1_test_labels),
    "trained CNN": representation_metrics(trained_train, c1_train_labels, trained_test, c1_test_labels),
}
print({name: round(report["accuracy"], 4) for name, report in c1_reports.items()})
save_course_figure(plot_representation_metrics(c1_reports, class_names), "c1_metrics.png")
# Reuse B4's query indices and test gallery, not its raw-pixel panel.
assert torch.equal(c1_test_labels, b4_labels), "C1/B4 test labels are not aligned"
assert torch.equal(c1_test_images, b4_images), "C1/B4 test image order changed"
c1_retrieval_features = {
    "HOG": hog_test,
    "untrained TinyResNet": random_test,
    "trained TinyResNet": trained_test,
}
c1_figure = plot_retrieval(
    c1_test_images, c1_test_labels, c1_retrieval_features, b4_query_indices,
    nearest_neighbors, 5,
)
c1_figure.suptitle("C1 HOG / untrained / trained: B4 queries and self-excluded test gallery")
save_course_figure(c1_figure, "c1_retrieval.png")

### 9.1. Student evidence — C1

State one semantic improvement, one useful low-level case, and why the same queries were used: **[replace]**.

## 10. C2 — Nuisance stress tests

Before execution, write directional predictions for brightness, blur, and central occlusion (or three approved replacements): **[replace before execution]**.

In [ ]:
c2_images, c2_labels = collect_loader(test_loader, 32)
c2_plans = {"brightness": [1.0, .8, .6, .4], "blur": [0, .5, 1, 2],
            "occlusion": [0, .10, .25, .40]}
c2_records = nuisance_sweep(model, c2_images, c2_labels, c2_plans)
for kind, rows in c2_records.items():
    print(kind, [(row["severity"], round(row["accuracy"], 3),
                  round(row["confidence"], 3)) for row in rows])
save_course_figure(plot_nuisance_records(c2_records), "c2_nuisance_sweeps.png")

### 10.1. Student evidence — C2

Compare every observation to its prediction, show success/failure cases, and connect evidence to augmentation/receptive fields: **[replace]**.
The supplied plot shows sweeps and example transformations. Add labeled success/failure
examples (or state if none occur), and report the sample count.


## 11. C3 — Plain ablation, error, neighbors, and CAM

In [ ]:
def run_c3_model(factory, controlled_train_loader, controlled_val_loader):
    seed_everything(SEED)
    candidate = factory().to(DEVICE)
    optimizer = torch.optim.AdamW(candidate.parameters(), lr=3e-3, weight_decay=1e-4)
    rows = []
    for _ in range(1 if RUN_MODE == "smoke" else 5):
        train_result = train_one_epoch(candidate, controlled_train_loader, optimizer, DEVICE)
        val_result = evaluate(candidate, controlled_val_loader, DEVICE)
        rows.append((*train_result, val_result[0], val_result[1]))
    gradient_norm = math.sqrt(sum(float(p.grad.detach().pow(2).sum())
                                  for p in candidate.parameters() if p.grad is not None))
    return candidate, rows, gradient_norm

# The probe uses disposable clones; neither actual comparison stream is consumed.
c3_stream_evidence = verify_c3_stream_reset(train_loader, val_loader)
c3_residual_train, c3_residual_val = make_c3_controlled_loaders(train_loader, val_loader)
c3_plain_train, c3_plain_val = make_c3_controlled_loaders(train_loader, val_loader)
assert c3_residual_train is not c3_plain_train
assert c3_residual_train.dataset is c3_plain_train.dataset
assert torch.equal(c3_residual_train.generator.get_state(),
                   c3_plain_train.generator.get_state())
c3_residual, c3_residual_rows, c3_residual_grad = run_c3_model(
    lambda: TinyResNet(10), c3_residual_train, c3_residual_val)
c3_plain, c3_plain_rows, c3_plain_grad = run_c3_model(
    lambda: TinyPlainNet(10), c3_plain_train, c3_plain_val)
print({"data_stream_reset": c3_stream_evidence,
       "epochs": len(c3_residual_rows), "residual": c3_residual_rows,
       "plain": c3_plain_rows, "residual_gradient_norm": c3_residual_grad,
       "plain_gradient_norm": c3_plain_grad})

In [ ]:
wrong = torch.nonzero(test_predictions != test_labels, as_tuple=False).flatten()
if len(wrong):
    c3_index = int(wrong[test_confidence[wrong].argmax()]); c3_note = "highest-confidence error"
else:
    c3_index = int(test_confidence.argmin()); c3_note = "no test error: boundary case"
c3_neighbors, _ = nearest_neighbors(b4_trained[c3_index], b4_trained, 5,
                                    exclude_index=c3_index)
c3_cam = classifier_weight_cam(model, b4_images[c3_index:c3_index + 1].to(DEVICE))[0]
rgb = images_to_rgb01(b4_images[c3_index:c3_index + 1])[0].permute(1, 2, 0)
c3_cam = F.interpolate(
    c3_cam[None, None], size=tuple(rgb.shape[:2]), mode="bilinear", align_corners=False
)[0, 0]
figure, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(rgb); axes[0].set_title(c3_note)
axes[1].imshow(rgb); axes[1].imshow(c3_cam, cmap="jet", alpha=.45)
axes[1].set_title("CAM: correlational, not causal")
neighbor = int(c3_neighbors[0])
axes[2].imshow(images_to_rgb01(b4_images[neighbor:neighbor + 1])[0].permute(1, 2, 0))
axes[2].set_title(f"nearest embedding, y={int(b4_labels[neighbor])}")
for axis in axes: axis.axis("off")
figure.tight_layout()
save_course_figure(figure, "c3_error_neighbors_cam.png")
print({"truth": int(test_labels[c3_index]), "prediction": int(test_predictions[c3_index]),
       "confidence": float(test_confidence[c3_index]), "neighbors": c3_neighbors.tolist()})

### 11.1. Student evidence — C3

Include the printed seed/generator/first-augmented-batch reset evidence. Use the five-epoch standard comparison to revisit A4. Distinguish observed error/neighbors/CAM evidence from speculation: **[replace]**.

## 12. D1 — Images, sources, and splits

In Colab's Files pane, create `data/atlas_images/` beneath `ROOT` and upload your
images there (JPG, PNG, or WebP; convert HEIC first). Replace the example row in
`data/atlas_manifest.csv` with one row per image, using matching filenames and
`development` / `held-out` splits. Strip EXIF from the image copies before upload;
keep originals locally. Freeze the split before model selection.


In [ ]:
USE_APPROVED_ACCESSIBILITY_FALLBACK = False  # change only with advance staff approval
if USE_APPROVED_ACCESSIBILITY_FALLBACK:
    fallback_root = ROOT / "data" / "accessibility_fallback"
    approval_path = fallback_root / "APPROVAL_TEMPLATE.md"
    approval_report = validate_completed_approval_record(approval_path)
    if not approval_report["ok"]:
        raise RuntimeError(
            "The fallback is blocked until staff complete its approval record: "
            + "; ".join(approval_report["errors"])
        )
    atlas_manifest_path = fallback_root / "fallback_manifest.csv"
    atlas_image_dir = fallback_root / "images"
    enforce_own_photo = False  # permitted only by the completed explicit waiver above
    atlas_source = "staff-approved accessibility fallback"
else:
    atlas_manifest_path = ROOT / "data" / "atlas_manifest.csv"
    atlas_image_dir = ROOT / "data" / "atlas_images"
    enforce_own_photo = True
    atlas_source = "student collection"
atlas_report = read_and_validate_manifest(atlas_manifest_path, atlas_image_dir,
                                          enforce_own_photo=enforce_own_photo)
print(json.dumps({"source": atlas_source, "manifest": str(atlas_manifest_path),
                  "images": str(atlas_image_dir), "own_photo_enforced": enforce_own_photo,
                  "ok": atlas_report["ok"], "errors": atlas_report["errors"]}, indent=2))
if not atlas_report["ok"]:
    print("Fix the listed errors, strip EXIF from image copies, and rerun.")

### 12.1. Student evidence — D1

Use your own collection unless staff have completed
`data/accessibility_fallback/APPROVAL_TEMPLATE.md`; only then enable
`USE_APPROVED_ACCESSIBILITY_FALLBACK`. Describe your categories, anticipated
ambiguity, image sources and permissions, and how you chose the split: **[replace]**.


## 13. D2 — Pretrained features and development-set choices

The next cell prepares frozen features and development-set prototypes. Use only
development images to choose your method. Record your substantive decision and
planned intervention below **before** running held-out evaluation.


In [ ]:
if not atlas_report["ok"]:
    print("Complete D1 before running D2.")
else:
    atlas_extractor, atlas_dimension, atlas_weight_note, atlas_preprocess = make_atlas_extractor()
    atlas_tensors, atlas_labels, atlas_paths, atlas_splits, atlas_categories = load_atlas_images(
        atlas_report["rows"], atlas_image_dir, atlas_preprocess)
    atlas_features = extract_frozen_features(atlas_extractor, atlas_tensors)
    development = torch.tensor([split == "development" for split in atlas_splits])
    held_out = ~development
    atlas_prototypes, _ = cosine_prototypes(
        atlas_features[development], atlas_labels[development], len(atlas_categories))
    print({"extractor": atlas_weight_note, "development": int(development.sum()),
           "held_out": int(held_out.sum())})


### Freeze decisions before held-out evaluation

Development-only design decision, justification, and expected intervention effect:
**[replace before running the next cells]**. The brightness change below is an
example; justify it or replace it before evaluation. Choices made after viewing
held-out results must be labeled exploratory.


In [ ]:
# Declare before evaluating held-out images; replace or justify this example.
atlas_intervention_name = "10% brightness increase applied before weight preprocessing"
atlas_intervention = lambda image: ImageEnhance.Brightness(image).enhance(1.10)


In [ ]:
if not atlas_report["ok"]:
    print("Complete D1 and D2 setup before evaluating held-out images.")
else:
    atlas_predictions, atlas_scores = prototype_predict(atlas_features[held_out], atlas_prototypes)
    print({"extractor": atlas_weight_note, "dimension": atlas_dimension,
           "development": int(development.sum()), "held_out": int(held_out.sum()),
           "held_out_accuracy": float((atlas_predictions == atlas_labels[held_out]).float().mean()),
           "scores": "uncalibrated cosine similarities, not probabilities"})

### 13.1. Student evidence — D2

Justify extractor/layer and one substantive decision. State when held-out was evaluated and why scores are not probabilities: **[replace]**.

## 14. D3 — Self-contained Atlas HTML and Hero

In [ ]:
if not atlas_report["ok"]:
    print("Complete D1 and D2 before exporting the Atlas.")
else:
    held_rows = [row for row, split in zip(atlas_report["rows"], atlas_splits)
                 if split == "held-out"]
    held_paths = [path for path, split in zip(atlas_paths, atlas_splits)
                  if split == "held-out"]
    development_rows = [row for row, split in zip(atlas_report["rows"], atlas_splits)
                        if split == "development"]
    development_paths = [path for path, split in zip(atlas_paths, atlas_splits)
                         if split == "development"]
    held_features = atlas_features[held_out]
    development_features = atlas_features[development]
    held_tensors = atlas_tensors[held_out]

    # Apply the intervention declared before held-out evaluation.
    intervention_tensors, _, _, _, _ = load_atlas_images(
        atlas_report["rows"], atlas_image_dir, atlas_preprocess,
        rgb_intervention=atlas_intervention)
    intervention_features = extract_frozen_features(atlas_extractor, intervention_tensors)
    intervention_predictions, intervention_scores = prototype_predict(
        intervention_features[held_out], atlas_prototypes)
    held_truth = atlas_labels[held_out]
    before_accuracy = float((atlas_predictions == held_truth).float().mean())
    after_accuracy = float((intervention_predictions == held_truth).float().mean())
    accuracy_delta = after_accuracy - before_accuracy
    score_delta = float(intervention_scores.mean() - atlas_scores.mean())
    intervention_summary = (
        f"{atlas_intervention_name}. Held-out accuracy {before_accuracy:.3f} -> "
        f"{after_accuracy:.3f} (delta {accuracy_delta:+.3f}); mean top-prototype "
        f"cosine delta {score_delta:+.3f}."
    )

    success_index, evidence_index, evidence_kind = select_atlas_cases(
        held_rows, atlas_predictions, atlas_scores, atlas_categories)
    atlas_html_path = ARTIFACT_DIR / "recognition_atlas.html"
    hero_path = ARTIFACT_DIR / "hero_ANDREWID.png"
    gallery_path = ARTIFACT_DIR / "atlas_heldout_gallery.png"
    retrieval_path = ARTIFACT_DIR / "atlas_failure_retrieval.png"
    cam_path = ARTIFACT_DIR / "atlas_failure_prototype_cam.png"
    intervention_path = ARTIFACT_DIR / "atlas_intervention.png"
    plot_atlas_gallery(held_paths, held_rows, atlas_predictions, atlas_scores,
                       atlas_categories, gallery_path); plt.show()
    _, retrieval_indices, retrieval_scores = plot_atlas_retrieval(
        held_paths[evidence_index], held_rows[evidence_index], held_features[evidence_index],
        development_paths, development_rows, development_features, retrieval_path); plt.show()
    predicted_category = int(atlas_predictions[evidence_index])
    plot_atlas_prototype_cam(
        atlas_extractor, held_tensors[evidence_index], held_paths[evidence_index],
        atlas_prototypes[predicted_category], atlas_categories[predicted_category],
        cam_path, preprocess=atlas_preprocess); plt.show()
    plot_atlas_intervention(
        held_paths, held_rows, atlas_categories, atlas_predictions, atlas_scores,
        intervention_predictions, intervention_scores, atlas_intervention,
        intervention_path); plt.show()
    success_summary = (
        f"Success case index {success_index}"
        if success_index is not None else "No held-out success was observed"
    )
    retrieval_caption = (
        f"{success_summary}; {evidence_kind} index {evidence_index}. "
        f"The panel shows its five nearest development images with cosine scores."
    )
    cam_caption = (
        f"Classifier-weight CAM for the predicted prototype on the {evidence_kind}; "
        "shown on the exact model-input crop; it is correlational evidence, not a causal explanation."
    )
    limitation = (
        "This finite held-out set supports conclusions only for the documented categories, "
        "sources, and capture conditions; retrieval and CAM do not establish causality."
    )
    export_atlas_html(
        ROOT / "common" / "atlas_template.html", atlas_html_path,
        categories=atlas_categories, held_rows=held_rows, held_paths=held_paths,
        predictions=atlas_predictions, scores=atlas_scores, weight_note=atlas_weight_note,
        gallery_path=gallery_path, retrieval_path=retrieval_path, cam_path=cam_path,
        intervention_path=intervention_path, retrieval_caption=retrieval_caption,
        cam_caption=cam_caption, intervention_summary=intervention_summary,
        limitation=limitation)
    plot_atlas_hero(held_paths, held_rows, atlas_predictions, atlas_scores,
                    atlas_categories, hero_path); plt.show()
    page = atlas_html_path.read_text(encoding="utf-8")
    minimum_images = len(held_paths) + 4 + int(success_index is not None)
    assert "[replace" not in page.lower() and page.count("data:image") >= minimum_images
    if success_index is None:
        assert 'data-evidence="no-success"' in page and "No held-out success was observed." in page
        assert 'data-evidence="selected-success"' not in page
    else:
        assert 'data-evidence="selected-success"' in page and "Clear success:" in page
    print({"offline_html": str(atlas_html_path), "hero": str(hero_path),
           "evidence_assets": [str(gallery_path), str(retrieval_path), str(cam_path),
                               str(intervention_path)],
           "retrieved_development_indices": retrieval_indices.tolist(),
           "retrieval_scores": [round(float(value), 4) for value in retrieval_scores],
           "measured_intervention": intervention_summary})

### 14.1. Student evidence — D3

Open the HTML offline. Check the predictions, retrieval gallery, CAM, before/after
results, captions, and Hero preview. Explain what the intervention shows; label
any choices made after viewing held-out results as exploratory: **[replace]**.


## 15. D4 — Comprehension reflection

Write 150–250 words answering all prompts. If no generative AI was used, state that and analyze the most important supplied/independent component and its validation.

**Student response: [replace]**

## 16. Submission audit

- [ ] Public tests pass against `student.pa2`.
- [ ] Standard mode ran in a fresh Colab runtime with the required data and 25 epochs.
- [ ] Best checkpoint reloads identically; every graded plot is in the PDF.
- [ ] B4/C1 share queries; C2 predictions precede execution; C3 standard is five epochs.
- [ ] Manifest passes, held-out is frozen, exact weight preprocessing is used, HTML works offline.
- [ ] EXIF/private data removed; collaborators, sources, weights, data, and AI disclosed.

Use the backup cell below before disconnecting or switching runtimes. Restore
`student/pa2.py` after setup in a fresh runtime. Download the executed notebook
separately via **File → Download → Download .ipynb**; saving it does not preserve
separate runtime files. Keep source Atlas images locally and re-upload them for D1.

For submission names and inserting saved figures into the PDF, see the README.


In [ ]:
# SUPPLIED: backup code, manifests, and generated artifacts, not source photos/caches.
backup_path = ROOT / "pa2_work_backup.zip"
backup_files = [ROOT / "student/pa2.py", ROOT / "data/atlas_manifest.csv",
                ROOT / "answer_template.tex"]
if globals().get("USE_APPROVED_ACCESSIBILITY_FALLBACK", False):
    backup_files += [atlas_manifest_path, approval_path]
backup_files += sorted((ROOT / "artifacts").rglob("*"))
with zipfile.ZipFile(backup_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in backup_files:
        if path.is_file() and path.suffix.lower() != ".zip":
            archive.write(path, path.relative_to(ROOT).as_posix())
print("Backup (not the submission bundle):", backup_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(backup_path))
print("Download the executed notebook separately: File → Download → Download .ipynb.")
